# DCGAN on the Anime Faces dataset (Kaggle)Same structure as the Fashion-MNIST DCGAN, adapted for 64x64 RGB anime face images.

### Imports

In [ ]:
import osimport globimport numpy as npimport matplotlib.pyplot as pltimport tensorflow as tffrom tensorflow import kerasfrom tensorflow.keras import layers

### Download the dataset from KaggleThis downloads the ["Anime Faces"](https://www.kaggle.com/datasets/soumikrakshit/anime-faces) dataset (21,551 images, 64x64 RGB).You need `kagglehub` installed (`pip install kagglehub`). If the dataset requires authentication, set up your Kaggle API token first (`kaggle.json` in `~/.kaggle/`).

In [ ]:
import kagglehubdataset_path = kagglehub.dataset_download("soumikrakshit/anime-faces")print("Dataset downloaded to:", dataset_path)

### Load and preprocess the imagesFinds every image file under the downloaded folder, resizes it to 64x64, and scales pixel values to [0, 1](same [0, 1] / sigmoid convention we used for Fashion-MNIST).

In [ ]:
from PIL import ImageIMG_SIZE = 64image_paths = glob.glob(os.path.join(dataset_path, "**", "*.png"), recursive=True)image_paths += glob.glob(os.path.join(dataset_path, "**", "*.jpg"), recursive=True)print(f"Found {len(image_paths)} images")images = []for path in image_paths:    img = Image.open(path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))    images.append(np.array(img))X_train = np.array(images, dtype="float32") / 255.0print("X_train shape:", X_train.shape)

### Build the generatorSame pattern as before: Dense + reshape into a small deep feature map, then Conv2DTranspose blocksthat each double the spatial size and shrink the channel depth, ending in a sigmoid output.`100 -> 8x8x256 -> 16x16x128 -> 32x32x64 -> 64x64x3`

In [ ]:
latent_dim = 100generator = keras.Sequential([    layers.Dense(8*8*256, input_dim=latent_dim),    layers.BatchNormalization(),    layers.LeakyReLU(),    layers.Reshape((8, 8, 256)),    # 8x8x256 -> 16x16x128    layers.Conv2DTranspose(128, kernel_size=4, strides=2, padding="same"),    layers.BatchNormalization(),    layers.LeakyReLU(),    # 16x16x128 -> 32x32x64    layers.Conv2DTranspose(64, kernel_size=4, strides=2, padding="same"),    layers.BatchNormalization(),    layers.LeakyReLU(),    # 32x32x64 -> 64x64x3    layers.Conv2DTranspose(        3, kernel_size=4, strides=2, padding="same", activation="sigmoid"    )])generator.summary()

### Build the discriminatorMirrors the generator: Conv2D blocks that each halve the spatial size and grow the channel depth,ending in Flatten + a single sigmoid neuron.`64x64x3 -> 32x32x64 -> 16x16x128 -> 8x8x256 -> 1`

In [ ]:
discriminator = keras.Sequential([    # 64x64x3 -> 32x32x64    layers.Conv2D(64, kernel_size=4, strides=2, padding="same", input_shape=(64, 64, 3)),    layers.LeakyReLU(),    layers.Dropout(0.3),    # 32x32x64 -> 16x16x128    layers.Conv2D(128, kernel_size=4, strides=2, padding="same"),    layers.LeakyReLU(),    layers.Dropout(0.3),    # 16x16x128 -> 8x8x256    layers.Conv2D(256, kernel_size=4, strides=2, padding="same"),    layers.LeakyReLU(),    layers.Dropout(0.3),    # Convert feature maps into one vector    layers.Flatten(),    # Real/Fake prediction    layers.Dense(1, activation="sigmoid")])discriminator.summary()

### Compile the discriminator

In [ ]:
discriminator.compile(    optimizer=keras.optimizers.Adam(learning_rate=0.0002),    loss="binary_crossentropy",    metrics=["accuracy"])

### Combine generator and discriminator into the GAN

In [ ]:
discriminator.trainable = Falsegan_input = keras.Input(shape=(latent_dim,))fake_image = generator(gan_input)gan_output = discriminator(fake_image)gan = keras.Model(gan_input, gan_output)gan.compile(    optimizer=keras.optimizers.Adam(learning_rate=0.0002),    loss="binary_crossentropy",    metrics=["accuracy"])gan.summary()

### Train the DCGANSame alternating loop as before: train the discriminator on a batch of real + fake images,then train the generator (through the frozen-discriminator GAN model).Every 2000 steps it also plots a few generated images, so you can watch the noise turn into facesover time instead of only checking at the very end.

In [ ]:
epochs = 20000batch_size = 128for epoch in range(epochs):    # Train Discriminator    idx = np.random.randint(0, X_train.shape[0], batch_size)    real_images = X_train[idx]    noise = np.random.normal(0, 1, (batch_size, latent_dim))    fake_images = generator.predict(noise, verbose=0)    d_loss_real = discriminator.train_on_batch(real_images, np.ones((batch_size, 1)))    d_loss_fake = discriminator.train_on_batch(fake_images, np.zeros((batch_size, 1)))    # Train Generator (via the combined GAN model)    noise = np.random.normal(0, 1, (batch_size, latent_dim))    g_loss = gan.train_on_batch(noise, np.ones((batch_size, 1)))    if epoch % 500 == 0:        print(f"Epoch: {epoch} | "              f"D_loss: {d_loss_real[0]:.4f} | "              f"G_loss: {g_loss[0]:.4f}"        )    if epoch % 2000 == 0:        preview_noise = np.random.normal(0, 1, (5, latent_dim))        preview_images = generator.predict(preview_noise, verbose=0)        plt.figure(figsize=(10, 2))        for i in range(5):            plt.subplot(1, 5, i + 1)            plt.imshow(preview_images[i])            plt.axis("off")        plt.suptitle(f"Epoch {epoch}")        plt.show()

### Generate new images

In [ ]:
noise = np.random.normal(0, 1, (10, latent_dim))generated_images = generator.predict(noise, verbose=0)

### Display generated images

In [ ]:
plt.figure(figsize=(10, 2))for i in range(10):    plt.subplot(1, 10, i + 1)    plt.imshow(generated_images[i])    plt.axis("off")plt.tight_layout()plt.show()